In [26]:
import pandas as pd
import numpy as np
import optuna
import shap
import xgboost as xgb
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import brier_score_loss, average_precision_score, make_scorer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import GroupKFold, cross_validate
import miceforest as mf
import warnings, os, datetime

warnings.filterwarnings('ignore')

# ════════════════════════════════════════════════════════
# USER CONFIG — SET THESE BEFORE RUNNING
# ════════════════════════════════════════════════════════
FEATURES_TO_SELECT = ['Age', 'Sex', 'Рост', 'Вес', 'ЧСС (b)', 'Систолическое АД(b)', 'P', 'PQ', 'RR', 'QT', 'QRS', 'Инфаркт миокарда в анамнезе (<3)', 'Инфаркт миокарда в анамнезе (>3)', 'Инфаркт миокарда со стентированием в анамнезе', 'ОНМК (иш) в анамнезе', 'ОНМК (гем) в анамнезе', 'Стентирование в анамнезе', 'ФП a (в анамнезе)','Открытая перация на сердце в анамнезе']
MICE_COLUMNS = ['Age', 'Sex', 'Рост', 'Вес', 'ЧСС (b)', 'Систолическое АД(b)', 'P', 'PQ', 'RR', 'QT', 'QRS', 'Инфаркт миокарда в анамнезе (<3)', 'Инфаркт миокарда в анамнезе (>3)', 'Инфаркт миокарда со стентированием в анамнезе', 'ОНМК (иш) в анамнезе', 'ОНМК (гем) в анамнезе', 'Стентирование в анамнезе', 'ФП a (в анамнезе)','Открытая перация на сердце в анамнезе']  # e.g. ['SpO2', 'Глюкоза(a)', ...]
TARGET = 'КШ развился в реанимации'
N_FOLDS = 5
OPTUNA_TRIALS = 300
RANDOM_STATE = 42


In [ ]:
# ════════════════════════════════════════════════════════

# 1. Load & merge

# ════════════════════════════════════════════════════════

dataAllFull = pd.read_excel('data_raw/side_project/DataSet_V49_9_03_26.xlsx')

data = dataAllFull.loc[

    (dataAllFull['STEMI (Новый)'] == 'Да') &

    (dataAllFull['Наличие в файле'] == 'Да') &

    (dataAllFull['ЧКВ'] == 'Да')

].copy()



print(f'Filtered v49: {data.shape}')



killip_df = pd.read_excel('data_raw/side_project/filtered_killip.xlsx')

killip_df = killip_df.drop(columns='Unnamed: 0', errors='ignore')



key = 'Код пациента'

common_cols = set(data.columns) & set(killip_df.columns)

killip_sub = killip_df[[key] + [c for c in killip_df.columns if c not in common_cols]].copy()

final_df = data.merge(killip_sub, on=key, how='inner')

print(f'Merged: {final_df.shape}')

print(f'Target: {final_df[TARGET].value_counts().to_dict()}')



# ════════════════════════════════════════════════════════

# 2. MICE imputation

# ════════════════════════════════════════════════════════

mice_cols = [m for m in MICE_COLUMNS if m in final_df.columns and pd.api.types.is_numeric_dtype(final_df[m])]

if MICE_COLUMNS and mice_cols:

    kernel = mf.ImputationKernel(data=final_df[mice_cols], random_state=RANDOM_STATE)

    kernel.mice(iterations=10)


    final_df[mice_cols] = kernel.complete_data(dataset=0).values

    print('MICE done.')


# ════════════════════════════════════════════════════════
# 3. Prepare X, y
# ════════════════════════════════════════════════════════

# Convert all object columns to category
to_cat = final_df.select_dtypes(include=['object']).columns
final_df[to_cat] = final_df[to_cat].astype('category')
print(f'Converted {len(to_cat)} object columns → category')

features = [f for f in FEATURES_TO_SELECT if f in final_df.columns]
X_orig = final_df[features].copy()
y = final_df[TARGET].astype(int)
groups = final_df[key]
n_orig = len(features)
num_cols = X_orig.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = X_orig.select_dtypes(include=['object', 'category']).columns.tolist()
print(f'Features ({n_orig}): {num_cols} + {cat_cols}')

if n_orig == 0:
    raise ValueError('FEATURES_TO_SELECT is empty! Set it in Cell 0 before running.')





# ════════════════════════════════════════════════════════
# 4. Preprocessing — fit ONCE on full data
# ════════════════════════════════════════════════════════

lr_pipe = Pipeline([('pre', ColumnTransformer([
    ('num', StandardScaler(), num_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols),
]))])


xgb_pipe = Pipeline([('pre', ColumnTransformer([
    ('num', 'passthrough', num_cols),
    ('cat', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), cat_cols),
]))])


X_lr = lr_pipe.fit_transform(X_orig)
X_xgb = xgb_pipe.fit_transform(X_orig)

# Feature names for mapping OHE columns → original features
lr_all_names = list(lr_pipe.named_steps['pre'].get_feature_names_out())
xgb_all_names = [f'num__{c}' for c in num_cols] + [f'cat__{c}' for c in cat_cols]

print(f'X_lr: {X_lr.shape}, X_xgb: {X_xgb.shape}')



# ════════════════════════════════════════════════════════
# 5. Build mapping: original feature index → columns in X_lr / X_xgb
# ════════════════════════════════════════════════════════

# OHE expands one categorical to N binary columns.
# RFE/Greedy operate on ORIGINAL features using this mapping.



def build_mapping(names, orig_features):
    mapping = {i: [] for i in range(len(orig_features))}
    for col_idx, name in enumerate(names):
        for orig_idx, feat in enumerate(orig_features):
            if name.startswith(f'num__{feat}') or name.startswith(f'cat__{feat}'):
                mapping[orig_idx].append(col_idx)
                break
    return mapping



orig_to_lr = build_mapping(lr_all_names, features)
orig_to_xgb = build_mapping(xgb_all_names, features)

print('Mapping:')

for i in range(n_orig):
    print(f'  {features[i]} → LR:{orig_to_lr[i]}, XGB:{orig_to_xgb[i]}')

print('Preprocessing complete.')

Filtered v49: (7524, 473)
Merged: (5588, 510)
Target: {0: 5406, 1: 182}


In [ ]:
# ════════════════════════════════════════════════════════
# Baseline: Optuna tuning on ALL features
# ════════════════════════════════════════════════════════

gkf = GroupKFold(n_splits=N_FOLDS)
scoring = {
    'roc_auc': 'roc_auc',
    'pr_auc': make_scorer(average_precision_score),
    'brier': make_scorer(brier_score_loss, needs_threshold=True),
    'precision': 'precision',
    'recall': 'recall',
    'f1': 'f1',
}

def tune(X_data, model_type):
    def objective(trial):
        if model_type == 'lr':
            params = {
                'C': trial.suggest_float('C', 1e-3, 1e2, log=True),
                'penalty': 'l2',
                'solver': 'lbfgs',
                'max_iter': 1000,
                'class_weight': 'balanced'
            }
        else:
            params = {
                'n_estimators': trial.suggest_int('n_estimators', 100, 500, step=50),
                'max_depth': trial.suggest_int('max_depth', 3, 8),
                'learning_rate': trial.suggest_float('lr', 1e-3, 0.3, log=True),
                'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
                'reg_alpha': trial.suggest_float('alpha', 0, 1.0),
                'reg_lambda': trial.suggest_float('lambda', 0, 1.0),
                'random_state': RANDOM_STATE,
                'class_weight': 'balanced'
            }

        clf = LogisticRegression(**params) if model_type == 'lr' else xgb.XGBClassifier(**params)
        sc = cross_validate(clf, X_data, y, groups=groups, cv=gkf, scoring='roc_auc')
        return sc['test_score'].mean()

    study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
    study.optimize(objective, n_trials=OPTUNA_TRIALS, show_progress_bar=False)
    return study.best_params, study.best_value

print('=== BASELINE ===')

lr_bparams, lr_bval = tune(X_lr, 'lr')
xgb_bparams, xgb_bval = tune(X_xgb, 'xgb')
print(f'LR ROC-AUC: {lr_bval:.4f}, XGB ROC-AUC: {xgb_bval:.4f}')

# ════════════════════════════════════════════════════════
# RFE — ranks ORIGINAL features, evaluates on all OHE columns
# ════════════════════════════════════════════════════════

def run_rfe(X_data, best_params, model_type, orig_to_cols):
    orig_to_col = orig_to_cols
    n_orig = len(orig_to_col)
    print(f'  RFE ({model_type}): {n_orig} original features')

    # Fit full model → per-original-feature importance
    clf_full = (LogisticRegression(**best_params, max_iter=1000) if model_type == 'lr' else xgb.XGBClassifier(**best_params, random_state=RANDOM_STATE))
    clf_full.fit(X_data, y)

    if model_type == 'lr':
        coefs = np.abs(clf_full.coef_[0])
        orig_imps = np.array([max(coefs[cols]) if cols else 0 for cols in orig_to_col.values()])
    else:
        imps = clf_full.feature_importances_
        orig_imps = np.array([sum(imps[cols]) for cols in orig_to_col.values()])

    ranked = np.argsort(-orig_imps)
    print(f'    Importances: {dict(zip([features[i] for i in ranked], orig_imps[ranked].round(3)))}')
    rows = []

    for k in range(1, n_orig + 1):
        orig_keep = ranked[:k]
        cols_keep = [c for i in orig_keep for c in orig_to_col[i]]
        clf = (LogisticRegression(**best_params, max_iter=1000) if model_type == 'lr' else xgb.XGBClassifier(**best_params, random_state=RANDOM_STATE))
        sc = cross_validate(clf, X_data[:, cols_keep], y, groups=groups, cv=gkf, scoring=scoring)
        ms = {k2: np.mean(v2) for k2, v2 in sc.items()}
        fnames = [features[i] for i in orig_keep]
        rows.append({'Method': f'RFE_{model_type.upper()}', 'Step': k, 'N_Features': k,
                     'Features': str(fnames), 'ROC_AUC': ms['test_roc_auc'],
                     'PR_AUC': ms['test_pr_auc'], 'Brier': ms['test_brier'],
                     'Precision': ms['test_precision'], 'Recall': ms['test_recall'],
                     'F1': ms['test_f1']})
        print(f'    k={k:2d}: ROC-AUC={ms["test_roc_auc"]:.4f}  {fnames}')
    return pd.DataFrame(rows)

print('=== RFE ===')
rfe_lr = run_rfe(X_lr, lr_bparams, 'lr', orig_to_lr)
rfe_xgb = run_rfe(X_xgb, xgb_bparams, 'xgb', orig_to_xgb)
print('RFE complete.')

[I 2026-08-03 08:07:51,174] A new study created in memory with name: no-name-5fb94d89-f510-4083-affb-18901c59ae3a
[W 2026-08-03 08:07:51,207] Trial 0 failed with parameters: {'C': 0.0745934328572655} because of the following error: ValueError('\nAll the 5 fits failed.\nIt is very likely that your model is misconfigured.\nYou can try to debug the error by setting error_score=\'raise\'.\n\nBelow are more details about the failures:\n--------------------------------------------------------------------------------\n5 fits failed with the following error:\nTraceback (most recent call last):\n  File "/opt/conda/lib/python3.11/site-packages/sklearn/model_selection/_validation.py", line 833, in _fit_and_score\n    estimator.fit(X_train, y_train, **fit_params)\n  File "/opt/conda/lib/python3.11/site-packages/sklearn/base.py", line 1336, in wrapper\n    return fit_method(estimator, *args, **kwargs)\n           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^\n  File "/opt/conda/lib/python3.11/site-package

=== BASELINE ===


ValueError: 
All the 5 fits failed.
It is very likely that your model is misconfigured.
You can try to debug the error by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
5 fits failed with the following error:
Traceback (most recent call last):
  File "/opt/conda/lib/python3.11/site-packages/sklearn/model_selection/_validation.py", line 833, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "/opt/conda/lib/python3.11/site-packages/sklearn/base.py", line 1336, in wrapper
    return fit_method(estimator, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/conda/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py", line 1191, in fit
    X, y = validate_data(
           ^^^^^^^^^^^^^^
  File "/opt/conda/lib/python3.11/site-packages/sklearn/utils/validation.py", line 2919, in validate_data
    X, y = check_X_y(X, y, **check

In [ ]:
# ════════════════════════════════════════════════════════
# Greedy Forward & Backward — on ORIGINAL features
# ════════════════════════════════════════════════════════
def greedy(X_data, best_params, model_type, orig_to_cols, direction):
    orig_to_col = orig_to_cols
    n_orig = len(orig_to_col)
    dname = 'Forward' if direction == 'fwd' else 'Backward'
    print(f'  Greedy {dname} ({model_type}): {n_orig} original features')

    rows = []
    selected = [] if direction == 'fwd' else list(range(n_orig))

    for step in range(1, n_orig + 1):
        best_score = -np.inf if direction == 'fwd' else np.inf
        best_feat = None

        for f_orig in range(n_orig):
            if direction == 'fwd' and f_orig in selected:
                continue
            if direction == 'bwd' and f_orig not in selected:
                continue
            sub_orig = selected + [f_orig] if direction == 'fwd' else [x for x in selected if x != f_orig]
            if not sub_orig:
                continue
            cols = [c for i in sub_orig for c in orig_to_col[i]]
            tmp = (LogisticRegression(**best_params, max_iter=1000) if model_type == 'lr'
                   else xgb.XGBClassifier(**best_params, random_state=RANDOM_STATE))
            ms_val = np.mean(cross_validate(tmp, X_data[:, cols], y, groups=groups,
                                            cv=gkf, scoring='roc_auc')['test_score'])
            if (direction == 'fwd' and ms_val > best_score) or (direction == 'bwd' and ms_val < best_score):
                best_score, best_feat = ms_val, f_orig

        if best_feat is None:
            break
        if direction == 'fwd':
            selected.append(best_feat)
        else:
            selected.remove(best_feat)
        if not selected:
            break

        cols = [c for i in selected for c in orig_to_col[i]]
        tmp = (LogisticRegression(**best_params, max_iter=1000) if model_type == 'lr'
               else xgb.XGBClassifier(**best_params, random_state=RANDOM_STATE))
        sc = cross_validate(tmp, X_data[:, cols], y, groups=groups, cv=gkf, scoring=scoring)
        ms = {k2: np.mean(v2) for k2, v2 in sc.items()}
        fnames = [features[i] for i in selected]
        action = 'added' if direction == 'fwd' else 'removed'
        print(f'    Step {step}/{n_orig}: {action} {features[best_feat]}, ROC-AUC={ms["test_roc_auc"]:.4f}')
        rows.append({'Method': f'Greedy_{dname}_{model_type.upper()}', 'Step': step,
                     'N_Features': len(selected), 'Features': str(fnames),
                     'ROC_AUC': ms['test_roc_auc'], 'PR_AUC': ms['test_pr_auc'],
                     'Brier': ms['test_brier'], 'Precision': ms['test_precision'],
                     'Recall': ms['test_recall'], 'F1': ms['test_f1']})
    return pd.DataFrame(rows)

print('=== GREEDY ===')
fwd_lr = greedy(X_lr, lr_bparams, 'lr', orig_to_lr, 'fwd')
bwd_lr = greedy(X_lr, lr_bparams, 'lr', orig_to_lr, 'bwd')
fwd_xgb = greedy(X_xgb, xgb_bparams, 'xgb', orig_to_xgb, 'fwd')
bwd_xgb = greedy(X_xgb, xgb_bparams, 'xgb', orig_to_xgb, 'bwd')
print('Greedy complete.')

# ════════════════════════════════════════════════════════
# Save results
# ════════════════════════════════════════════════════════
all_res = pd.concat([rfe_lr, rfe_xgb, fwd_lr, bwd_lr, fwd_xgb, bwd_xgb], ignore_index=True)
print(f'\nAll results: {len(all_res)} rows')
os.makedirs('results', exist_ok=True)
ts = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
all_res.to_csv(f'results/feature_selection_{ts}.csv', index=False)
print(f'Saved: results/feature_selection_{ts}.csv')

# Best per method
print('\nBest per method:')
for method in all_res['Method'].unique():
    bi = all_res[all_res['Method'] == method]['ROC_AUC'].idxmax()
    print(f'  {method}: k={all_res.loc[bi,"Step"]} ROC-AUC={all_res.loc[bi,"ROC_AUC"]:.4f}')
ob = all_res.loc[all_res['ROC_AUC'].idxmax()]
print(f'\nOverall best: {ob["Method"]} k={ob["Step"]} ROC-AUC={ob["ROC_AUC"]:.4f}')

# ════════════════════════════════════════════════════════
# Train best model + SHAP / coefficients
# ════════════════════════════════════════════════════════
bmodel = ob['Method'].split('_')[-1]
bfeat_names = eval(ob['Features'])
Xb = X_orig[bfeat_names]
Xb_num = Xb.select_dtypes(include=['number']).columns.tolist()
Xb_cat = Xb.select_dtypes(include=['object', 'category']).columns.tolist()

if bmodel == 'LR':
    best_pipe = Pipeline([('pre', ColumnTransformer([
        ('num', StandardScaler(), Xb_num),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), Xb_cat),
    ]))])
    base_clf = LogisticRegression(**lr_bparams, max_iter=1000)
else:
    best_pipe = Pipeline([('pre', ColumnTransformer([
        ('num', 'passthrough', Xb_num),
        ('cat', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), Xb_cat),
    ]))])
    base_clf = xgb.XGBClassifier(**xgb_bparams, random_state=RANDOM_STATE)

Xb_pre = best_pipe.fit_transform(Xb)
base_clf.fit(Xb_pre, y)
bsc = cross_validate(base_clf, Xb_pre, y, groups=groups, cv=gkf, scoring=scoring)
print('\nFinal CV scores:')
for k2, v2 in bsc.items():
    print(f'  {k2}: {np.mean(v2):.4f} +/- {np.std(v2):.4f}')

# Feature importance
if bmodel == 'XGB':
    expl = shap.TreeExplainer(base_clf)
    sv = expl.shap_values(Xb_pre)
    shap.summary_plot(sv, Xb_pre, plot_type='bar', show=False)
    fi = pd.DataFrame({'Feature': Xb.columns, 'SHAP': np.abs(sv).mean(axis=0)}).sort_values('SHAP', ascending=False)
    print(fi.to_string(index=False))
else:
    fnames_list = best_pipe.named_steps['pre'].get_feature_names_out().tolist()
    fi = pd.DataFrame({'Feature': fnames_list, 'Coefficient': base_clf.coef_[0],
                       'Abs_Coeff': np.abs(base_clf.coef_[0])}).sort_values('Abs_Coeff', ascending=False)
    print(fi[['Feature', 'Coefficient']].to_string(index=False))

fi.to_csv(f'results/best_model_importance_{ts}.csv', index=False)
print(f'\nSaved: results/best_model_importance_{ts}.csv')
print('Done.')
